# OSRM + TomTom Matrix Builder  
## Distribusi Kosmetik Perkotaan

Notebook ini digunakan untuk membentuk **saved matrix** yang dibutuhkan oleh notebook utama.

**Tujuan utama**
1. Membaca data Excel sumber.
2. Membentuk `osrm_node_index.csv`.
3. Mengambil matriks jarak dan durasi dari OSRM untuk setiap instance harian.
4. Menyimpan matriks OSRM dalam format CSV.
5. Membentuk `tomtom_traffic_multiplier.csv` berdasarkan multiplier final p50, p85, dan p95.
6. Membentuk travel-time matrix p50, p85, dan p95.
7. Menyimpan seluruh file ke ZIP.

**Output utama**
- `osrm_node_index.csv`
- `tomtom_traffic_multiplier.csv`
- `osrm_matrices_by_date/`
- `travel_time_matrices_by_date/`
- `matrix_manifest.csv`
- `matrix_source_audit.csv`
- `osrm_tomtom_saved_matrices.zip`

**Catatan penting**  
Notebook ini membutuhkan koneksi internet jika memakai OSRM public endpoint. Untuk uji awal, jalankan hanya 1–3 tanggal terlebih dahulu. Setelah berhasil, baru jalankan semua tanggal.

In [ ]:
# ============================================================
# CELL 1 — SETUP ENVIRONMENT
# ============================================================

import os, re, json, time, math, zipfile, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path("/content")
OUTPUT_DIR = BASE_DIR / "osrm_tomtom_matrix_outputs"
OSRM_DIR = OUTPUT_DIR / "osrm_matrices_by_date"
TT_DIR = OUTPUT_DIR / "travel_time_matrices_by_date"
AUDIT_DIR = OUTPUT_DIR / "audit"
TEMPLATE_DIR = OUTPUT_DIR / "templates"

for d in [OUTPUT_DIR, OSRM_DIR, TT_DIR, AUDIT_DIR, TEMPLATE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)
print("Run date:", datetime.now())

In [ ]:
# ============================================================
# CELL 2 — CONFIGURATION UNTUK MEMBANGUN MATRIX SEMUA TANGGAL
# ============================================================

EXCEL_NAME = "baseline_dataset_prepared_coordinate_fixed.xlsx"
EXCEL_PATH = BASE_DIR / EXCEL_NAME

# OSRM endpoint.
# Untuk public endpoint:
OSRM_BASE_URL = "https://router.project-osrm.org"
OSRM_PROFILE = "driving"

# Untuk local OSRM server, ubah menjadi contoh:
# OSRM_BASE_URL = "http://localhost:5000"

RUN_OSRM_API = True

# ============================================================
# MODE FINAL: SEMUA TANGGAL
# ============================================================
# Jika SELECTED_DATES = None dan MAX_DATES_FOR_TEST = None,
# notebook akan memproses seluruh tanggal/instance pada dataset.
SELECTED_DATES = None
MAX_DATES_FOR_TEST = None

# Resume mode:
# Jika sempat gagal di tengah jalan, jalankan ulang notebook.
# Tanggal yang output matrix-nya sudah lengkap akan dilewati otomatis.
RESUME_SKIP_EXISTING = True

# Public OSRM server sebaiknya diberi jeda agar tidak terlalu agresif.
# Untuk semua tanggal, gunakan jeda lebih aman.
REQUEST_SLEEP_SEC = 1.5
MAX_RETRIES = 4
RETRY_SLEEP_SEC = 5.0
REQUEST_TIMEOUT_SEC = 180

# Jika request OSRM gagal:
# True  = berhenti agar error diketahui
# False = catat error dan lanjut tanggal berikutnya
STOP_ON_OSRM_ERROR = True

# TomTom-based traffic multiplier final.
# Nilai ini disimpan sebagai file tomtom_traffic_multiplier.csv.
TOMTOM_MULTIPLIER = {
    "p50": 1.0000,
    "p85": 1.1503,
    "p95": 1.3071,
}

# Prefix nama file ZIP akhir.
ZIP_NAME = "osrm_tomtom_saved_matrices.zip"

if not EXCEL_PATH.exists() and IN_COLAB:
    print("Silakan upload file Excel sumber:", EXCEL_NAME)
    uploaded = files.upload()

print("Excel exists:", EXCEL_PATH.exists(), EXCEL_PATH)
print("RUN_OSRM_API:", RUN_OSRM_API)
print("SELECTED_DATES:", SELECTED_DATES)
print("MAX_DATES_FOR_TEST:", MAX_DATES_FOR_TEST)
print("RESUME_SKIP_EXISTING:", RESUME_SKIP_EXISTING)
print("Mode:", "ALL DATES" if SELECTED_DATES is None and MAX_DATES_FOR_TEST is None else "LIMITED/TEST")

In [ ]:
# ============================================================
# CELL 3 — HELPER FUNCTIONS
# ============================================================

def safe_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")

def time_to_min(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    try:
        parts = s.split(":")
        return int(parts[0]) * 60 + int(parts[1])
    except Exception:
        try:
            t = pd.to_datetime(s).time()
            return t.hour * 60 + t.minute
        except Exception:
            return np.nan

def build_osrm_coordinate_string(nodes_df):
    # OSRM uses lon,lat order.
    return ";".join([
        f"{float(row.longitude):.6f},{float(row.latitude):.6f}"
        for row in nodes_df.itertuples(index=False)
    ])

def osrm_table_request(nodes_df, base_url=OSRM_BASE_URL, profile=OSRM_PROFILE):
    coords = build_osrm_coordinate_string(nodes_df)
    url = f"{base_url}/table/v1/{profile}/{coords}"
    params = {
        "annotations": "duration,distance"
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT_SEC)
            if r.status_code == 200:
                data = r.json()
                if "durations" not in data or "distances" not in data:
                    raise ValueError(f"OSRM response does not contain durations/distances. Keys: {list(data.keys())}")
                return data, r.url
            else:
                last_error = f"HTTP {r.status_code}: {r.text[:500]}"
                print(f"Attempt {attempt}/{MAX_RETRIES} failed:", last_error)
        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}/{MAX_RETRIES} exception:", last_error)

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_SLEEP_SEC)

    raise RuntimeError(f"OSRM request failed after {MAX_RETRIES} attempts. Last error: {last_error}")

def save_matrix_outputs(date_str, nodes, durations_sec, distances_m, request_url):
    node_ids = nodes["node_id"].astype(str).tolist()

    duration_min = pd.DataFrame(np.array(durations_sec, dtype=float) / 60.0, index=node_ids, columns=node_ids)
    distance_km = pd.DataFrame(np.array(distances_m, dtype=float) / 1000.0, index=node_ids, columns=node_ids)

    # Wide format
    distance_wide = distance_km.copy()
    duration_wide = duration_min.copy()
    distance_wide.insert(0, "node_id", distance_wide.index)
    duration_wide.insert(0, "node_id", duration_wide.index)

    distance_wide_path = OSRM_DIR / f"{date_str}_osrm_distance_matrix_km_wide.csv"
    duration_wide_path = OSRM_DIR / f"{date_str}_osrm_duration_matrix_min_wide.csv"

    distance_wide.to_csv(distance_wide_path, index=False, encoding="utf-8-sig")
    duration_wide.to_csv(duration_wide_path, index=False, encoding="utf-8-sig")

    # Long format
    dist_long = distance_km.stack().reset_index()
    dist_long.columns = ["origin_id", "destination_id", "distance_km"]

    dur_long = duration_min.stack().reset_index()
    dur_long.columns = ["origin_id", "destination_id", "duration_min"]

    distance_long_path = OSRM_DIR / f"{date_str}_osrm_distance_matrix_km_long.csv"
    duration_long_path = OSRM_DIR / f"{date_str}_osrm_duration_matrix_min_long.csv"

    dist_long.to_csv(distance_long_path, index=False, encoding="utf-8-sig")
    dur_long.to_csv(duration_long_path, index=False, encoding="utf-8-sig")

    # Travel-time matrix p50/p85/p95
    tt_paths = {}
    for q, mult in TOMTOM_MULTIPLIER.items():
        tt = duration_min * float(mult)

        tt_wide = tt.copy()
        tt_wide.insert(0, "node_id", tt_wide.index)
        tt_wide_path = TT_DIR / f"{date_str}_travel_time_matrix_{q}_wide.csv"
        tt_wide.to_csv(tt_wide_path, index=False, encoding="utf-8-sig")

        tt_long = tt.stack().reset_index()
        tt_long.columns = ["origin_id", "destination_id", f"travel_time_min_{q}"]
        tt_long_path = TT_DIR / f"{date_str}_travel_time_matrix_{q}_long.csv"
        tt_long.to_csv(tt_long_path, index=False, encoding="utf-8-sig")

        tt_paths[q] = {
            "wide": str(tt_wide_path),
            "long": str(tt_long_path),
            "avg_travel_time_min": float(tt.values[tt.values > 0].mean()) if np.any(tt.values > 0) else 0.0
        }

    # Node index per date
    node_index_path = OSRM_DIR / f"{date_str}_node_index.csv"
    nodes.to_csv(node_index_path, index=False, encoding="utf-8-sig")

    audit = {
        "date": date_str,
        "n_nodes": len(nodes),
        "request_url": request_url,
        "distance_wide_file": str(distance_wide_path),
        "duration_wide_file": str(duration_wide_path),
        "distance_long_file": str(distance_long_path),
        "duration_long_file": str(duration_long_path),
        "node_index_file": str(node_index_path),
        "avg_distance_km": float(distance_km.values[distance_km.values > 0].mean()) if np.any(distance_km.values > 0) else 0.0,
        "avg_duration_min": float(duration_min.values[duration_min.values > 0].mean()) if np.any(duration_min.values > 0) else 0.0,
        "max_duration_min": float(np.nanmax(duration_min.values)),
        "max_distance_km": float(np.nanmax(distance_km.values)),
        "travel_time_outputs": tt_paths
    }
    return audit

def create_tomtom_multiplier_file():
    df = pd.DataFrame({
        "quantile": list(TOMTOM_MULTIPLIER.keys()),
        "multiplier": list(TOMTOM_MULTIPLIER.values()),
        "interpretation": [
            "median / normal travel condition",
            "dense traffic condition",
            "highly congested travel condition"
        ],
        "source_note": [
            "saved final TomTom-based multiplier used in experiment",
            "saved final TomTom-based multiplier used in experiment",
            "saved final TomTom-based multiplier used in experiment"
        ]
    })
    path = OUTPUT_DIR / "tomtom_traffic_multiplier.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return df, path

print("Helper functions ready.")


def matrix_outputs_exist_for_date(date_str):
    required_paths = [
        OSRM_DIR / f"{date_str}_osrm_distance_matrix_km_wide.csv",
        OSRM_DIR / f"{date_str}_osrm_duration_matrix_min_wide.csv",
        OSRM_DIR / f"{date_str}_osrm_distance_matrix_km_long.csv",
        OSRM_DIR / f"{date_str}_osrm_duration_matrix_min_long.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p50_wide.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p85_wide.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p95_wide.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p50_long.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p85_long.csv",
        TT_DIR / f"{date_str}_travel_time_matrix_p95_long.csv",
        OSRM_DIR / f"{date_str}_node_index.csv",
    ]
    return all(p.exists() and p.stat().st_size > 0 for p in required_paths)


In [ ]:
# ============================================================
# CELL 4 — LOAD EXCEL DAN PREPROCESS CUSTOMER-DAY STOPS
# ============================================================

if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"File Excel belum ditemukan: {EXCEL_PATH}")

xl = pd.ExcelFile(EXCEL_PATH)
print("Sheets:", xl.sheet_names)

stops_raw = pd.read_excel(EXCEL_PATH, sheet_name="Stops")
depot = pd.read_excel(EXCEL_PATH, sheet_name="Depot")

stops = stops_raw.copy()
stops["first_delivery_dt"] = pd.to_datetime(stops["first_delivery_dt"], errors="coerce")
stops["delivery_date"] = stops["first_delivery_dt"].dt.date

for col in ["longitude", "latitude", "total_weight", "total_volume", "service_time_min_baseline"]:
    stops[col] = pd.to_numeric(stops[col], errors="coerce")

stops["tw_start_min"] = stops["tw_start"].apply(time_to_min)
stops["tw_end_min"] = stops["tw_end"].apply(time_to_min)

agg = stops.groupby(["delivery_date", "customer_code"], as_index=False).agg(
    customer_name=("customer_name", "first"),
    longitude=("longitude", "first"),
    latitude=("latitude", "first"),
    order_count=("order_number", "count"),
    total_weight=("total_weight", "sum"),
    total_volume=("total_volume", "sum"),
    quantity_order=("quantity_order", "sum"),
    tw_start_min=("tw_start_min", "min"),
    tw_end_min=("tw_end_min", "min"),
    service_time_min_baseline=("service_time_min_baseline", "max")
)

agg["delivery_date_str"] = pd.to_datetime(agg["delivery_date"]).dt.strftime("%Y-%m-%d")
agg.to_csv(AUDIT_DIR / "customer_day_stops_for_matrix_builder.csv", index=False, encoding="utf-8-sig")

depot_lon = float(depot.loc[0, "longitude"])
depot_lat = float(depot.loc[0, "latitude"])

display(stops.head())
display(agg.head())

print("Order awal:", len(stops))
print("Customer-day stop:", len(agg))
print("Jumlah tanggal:", agg["delivery_date_str"].nunique())

In [ ]:
# ============================================================
# CELL 5 — GENERATE OSRM NODE INDEX GLOBAL
# ============================================================

global_node_index = pd.concat([
    pd.DataFrame({
        "node_id": ["DEPOT_1"],
        "customer_code": ["DEPOT_1"],
        "customer_name": ["Depot"],
        "longitude": [depot_lon],
        "latitude": [depot_lat],
        "node_type": ["depot"]
    }),
    agg[["customer_code", "customer_name", "longitude", "latitude"]]
    .drop_duplicates("customer_code")
    .assign(
        node_id=lambda x: x["customer_code"].astype(str),
        node_type="customer"
    )[["node_id", "customer_code", "customer_name", "longitude", "latitude", "node_type"]]
], ignore_index=True)

global_node_index["node_id"] = global_node_index["node_id"].astype(str)

node_index_path = OUTPUT_DIR / "osrm_node_index.csv"
global_node_index.to_csv(node_index_path, index=False, encoding="utf-8-sig")

display(global_node_index.head())
print("Global node index saved:", node_index_path)
print("Jumlah node unik termasuk depot:", len(global_node_index))

In [ ]:
# ============================================================
# CELL 6 — GENERATE TOMTOM TRAFFIC MULTIPLIER FILE
# ============================================================

tomtom_df, tomtom_path = create_tomtom_multiplier_file()

display(tomtom_df)
print("TomTom multiplier saved:", tomtom_path)

In [ ]:
# ============================================================
# CELL 7 — PILIH SELURUH TANGGAL / INSTANCE UNTUK OSRM MATRIX
# ============================================================

all_dates = sorted(agg["delivery_date_str"].dropna().unique().tolist())

if SELECTED_DATES is not None:
    selected_dates = [d for d in all_dates if d in SELECTED_DATES]
else:
    selected_dates = all_dates.copy()

if MAX_DATES_FOR_TEST is not None:
    selected_dates = selected_dates[:MAX_DATES_FOR_TEST]

date_summary = []
for d in selected_dates:
    sub = agg[agg["delivery_date_str"] == d]
    already_done = matrix_outputs_exist_for_date(d)
    date_summary.append({
        "delivery_date": d,
        "n_customer_day_stops": len(sub),
        "n_nodes_with_depot": len(sub) + 1,
        "matrix_already_exists": already_done,
        "run_status": "skip_existing" if already_done and RESUME_SKIP_EXISTING else "will_process"
    })

date_summary_df = pd.DataFrame(date_summary)
date_summary_df.to_csv(AUDIT_DIR / "selected_dates_for_osrm_matrix.csv", index=False, encoding="utf-8-sig")

display(date_summary_df)
print("Total tanggal pada dataset:", len(all_dates))
print("Jumlah tanggal yang dipilih:", len(selected_dates))
print("Tanggal yang sudah punya matrix lengkap:", int(date_summary_df["matrix_already_exists"].sum()) if len(date_summary_df) else 0)
print("Tanggal yang akan diproses:", int((date_summary_df["run_status"] == "will_process").sum()) if len(date_summary_df) else 0)

In [ ]:
# ============================================================
# CELL 8 — RUN OSRM MATRIX BUILDER UNTUK SEMUA TANGGAL
# ============================================================

if not RUN_OSRM_API:
    raise RuntimeError("RUN_OSRM_API=False. Ubah menjadi True untuk mengambil matriks dari OSRM.")

audit_records = []
error_records = []
skipped_records = []

for idx, date_str in enumerate(selected_dates, start=1):
    display(Markdown(f"### Processing {idx}/{len(selected_dates)} — {date_str}"))

    if RESUME_SKIP_EXISTING and matrix_outputs_exist_for_date(date_str):
        msg = {
            "delivery_date": date_str,
            "status": "skipped_existing",
            "note": "Matrix files already complete for this date."
        }
        skipped_records.append(msg)
        print("SKIP:", msg)
        continue

    day_df = agg[agg["delivery_date_str"] == date_str].copy().reset_index(drop=True)

    nodes = pd.concat([
        pd.DataFrame({
            "node_id": ["DEPOT_1"],
            "customer_code": ["DEPOT_1"],
            "customer_name": ["Depot"],
            "longitude": [depot_lon],
            "latitude": [depot_lat],
            "node_type": ["depot"]
        }),
        day_df[["customer_code", "customer_name", "longitude", "latitude"]]
        .assign(
            node_id=lambda x: x["customer_code"].astype(str),
            node_type="customer"
        )[["node_id", "customer_code", "customer_name", "longitude", "latitude", "node_type"]]
    ], ignore_index=True)

    nodes["node_id"] = nodes["node_id"].astype(str)

    print("Nodes including depot:", len(nodes))

    try:
        data, request_url = osrm_table_request(nodes)
        audit = save_matrix_outputs(
            date_str=date_str,
            nodes=nodes,
            durations_sec=data["durations"],
            distances_m=data["distances"],
            request_url=request_url
        )
        audit["status"] = "processed"
        audit_records.append(audit)

        print("Saved matrices for:", date_str)
        print("Average duration min:", round(audit["avg_duration_min"], 3))
        print("Average distance km:", round(audit["avg_distance_km"], 3))

    except Exception as e:
        err = {
            "delivery_date": date_str,
            "n_nodes": len(nodes),
            "error": str(e)
        }
        error_records.append(err)
        print("ERROR:", err)

        if STOP_ON_OSRM_ERROR:
            pd.DataFrame(error_records).to_csv(AUDIT_DIR / "osrm_error_log.csv", index=False, encoding="utf-8-sig")
            if skipped_records:
                pd.DataFrame(skipped_records).to_csv(AUDIT_DIR / "osrm_skipped_existing_log.csv", index=False, encoding="utf-8-sig")
            raise

    time.sleep(REQUEST_SLEEP_SEC)

audit_df = pd.DataFrame([
    {k: v for k, v in rec.items() if k != "travel_time_outputs"}
    for rec in audit_records
])

audit_df.to_csv(AUDIT_DIR / "osrm_matrix_builder_audit.csv", index=False, encoding="utf-8-sig")

if skipped_records:
    pd.DataFrame(skipped_records).to_csv(AUDIT_DIR / "osrm_skipped_existing_log.csv", index=False, encoding="utf-8-sig")

if error_records:
    pd.DataFrame(error_records).to_csv(AUDIT_DIR / "osrm_error_log.csv", index=False, encoding="utf-8-sig")

display(audit_df)
print("Processed dates:", len(audit_records))
print("Skipped existing dates:", len(skipped_records))
print("Errors:", len(error_records))
print("Finished OSRM matrix builder.")

## Verifikasi Coverage Matrix Semua Tanggal

Cell berikut mengecek apakah matrix sudah terbentuk untuk seluruh tanggal/instance. Untuk output final, status coverage sebaiknya **complete**.

In [ ]:
# ============================================================
# CELL 8B — VERIFIKASI COVERAGE MATRIX SEMUA TANGGAL
# ============================================================

coverage_records = []
for d in all_dates:
    coverage_records.append({
        "delivery_date": d,
        "matrix_complete": matrix_outputs_exist_for_date(d),
        "n_nodes_with_depot": int(len(agg[agg["delivery_date_str"] == d]) + 1)
    })

coverage_all_df = pd.DataFrame(coverage_records)
coverage_all_df["status"] = np.where(coverage_all_df["matrix_complete"], "complete", "missing")
coverage_all_df.to_csv(AUDIT_DIR / "matrix_coverage_all_dates.csv", index=False, encoding="utf-8-sig")

coverage_summary = pd.DataFrame({
    "Komponen": [
        "Total instance dates",
        "Complete matrix dates",
        "Missing matrix dates",
        "Coverage percentage",
        "Coverage status"
    ],
    "Nilai": [
        len(coverage_all_df),
        int(coverage_all_df["matrix_complete"].sum()),
        int((~coverage_all_df["matrix_complete"]).sum()),
        round(coverage_all_df["matrix_complete"].mean() * 100, 2) if len(coverage_all_df) else 0,
        "complete" if coverage_all_df["matrix_complete"].all() else "partial"
    ]
})
coverage_summary.to_csv(AUDIT_DIR / "matrix_coverage_summary.csv", index=False, encoding="utf-8-sig")

display(coverage_summary)
display(coverage_all_df)

if not coverage_all_df["matrix_complete"].all():
    missing_dates = coverage_all_df.loc[~coverage_all_df["matrix_complete"], "delivery_date"].tolist()
    print("Tanggal yang belum lengkap:", missing_dates[:30], "..." if len(missing_dates) > 30 else "")
else:
    print("Semua tanggal sudah memiliki saved OSRM + TomTom matrix lengkap.")

In [ ]:
# ============================================================
# CELL 9 — BUILD COMBINED LONG MATRICES
# ============================================================

# Menggabungkan seluruh long matrix per tanggal menjadi file gabungan.
# File ini berguna jika notebook utama ingin membaca satu file long per jenis matrix.

combined_files = []

def combine_long_files(pattern, out_name):
    files_ = sorted(OSRM_DIR.glob(pattern))
    parts = []
    for p in files_:
        date_str = p.name.split("_")[0]
        df = pd.read_csv(p)
        df.insert(0, "delivery_date", date_str)
        parts.append(df)
    if parts:
        combined = pd.concat(parts, ignore_index=True)
        out_path = OUTPUT_DIR / out_name
        combined.to_csv(out_path, index=False, encoding="utf-8-sig")
        return out_path, len(combined)
    return None, 0

def combine_tt_long_files(q):
    files_ = sorted(TT_DIR.glob(f"*_travel_time_matrix_{q}_long.csv"))
    parts = []
    for p in files_:
        date_str = p.name.split("_")[0]
        df = pd.read_csv(p)
        df.insert(0, "delivery_date", date_str)
        parts.append(df)
    if parts:
        combined = pd.concat(parts, ignore_index=True)
        out_path = OUTPUT_DIR / f"combined_travel_time_matrix_{q}_long.csv"
        combined.to_csv(out_path, index=False, encoding="utf-8-sig")
        return out_path, len(combined)
    return None, 0

for pattern, out_name in [
    ("*_osrm_distance_matrix_km_long.csv", "combined_osrm_distance_matrix_km_long.csv"),
    ("*_osrm_duration_matrix_min_long.csv", "combined_osrm_duration_matrix_min_long.csv")
]:
    p, nrows = combine_long_files(pattern, out_name)
    combined_files.append({"file": out_name, "path": str(p), "n_rows": nrows})

for q in ["p50", "p85", "p95"]:
    p, nrows = combine_tt_long_files(q)
    combined_files.append({"file": f"combined_travel_time_matrix_{q}_long.csv", "path": str(p), "n_rows": nrows})

combined_df = pd.DataFrame(combined_files)
combined_df.to_csv(AUDIT_DIR / "combined_long_files_audit.csv", index=False, encoding="utf-8-sig")
display(combined_df)

In [ ]:
# ============================================================
# CELL 10 — PREVIEW OUTPUT MATRIX
# ============================================================

display(Markdown("## Ringkasan Output Matrix Builder"))

summary = pd.DataFrame({
    "Komponen": [
        "Global node index",
        "TomTom multiplier",
        "OSRM date node index files",
        "OSRM wide distance files",
        "OSRM wide duration files",
        "OSRM long distance files",
        "OSRM long duration files",
        "Travel time p50 wide files",
        "Travel time p85 wide files",
        "Travel time p95 wide files",
        "Audit files"
    ],
    "Jumlah": [
        1 if (OUTPUT_DIR / "osrm_node_index.csv").exists() else 0,
        1 if (OUTPUT_DIR / "tomtom_traffic_multiplier.csv").exists() else 0,
        len(list(OSRM_DIR.glob("*_node_index.csv"))),
        len(list(OSRM_DIR.glob("*_osrm_distance_matrix_km_wide.csv"))),
        len(list(OSRM_DIR.glob("*_osrm_duration_matrix_min_wide.csv"))),
        len(list(OSRM_DIR.glob("*_osrm_distance_matrix_km_long.csv"))),
        len(list(OSRM_DIR.glob("*_osrm_duration_matrix_min_long.csv"))),
        len(list(TT_DIR.glob("*_travel_time_matrix_p50_wide.csv"))),
        len(list(TT_DIR.glob("*_travel_time_matrix_p85_wide.csv"))),
        len(list(TT_DIR.glob("*_travel_time_matrix_p95_wide.csv"))),
        len(list(AUDIT_DIR.glob("*.csv"))),
    ]
})
display(summary)

# Preview file-file utama
display(Markdown("### Preview osrm_node_index.csv"))
display(pd.read_csv(OUTPUT_DIR / "osrm_node_index.csv").head(10))

display(Markdown("### Preview tomtom_traffic_multiplier.csv"))
display(pd.read_csv(OUTPUT_DIR / "tomtom_traffic_multiplier.csv"))

audit_path = AUDIT_DIR / "osrm_matrix_builder_audit.csv"
if audit_path.exists():
    display(Markdown("### Preview OSRM audit"))
    display(pd.read_csv(audit_path).head(20))

# Preview salah satu matrix
sample_dist_files = sorted(OSRM_DIR.glob("*_osrm_distance_matrix_km_wide.csv"))
sample_dur_files = sorted(OSRM_DIR.glob("*_osrm_duration_matrix_min_wide.csv"))
sample_tt_files = sorted(TT_DIR.glob("*_travel_time_matrix_p85_wide.csv"))

if sample_dist_files:
    display(Markdown(f"### Preview sample distance matrix: `{sample_dist_files[0].name}`"))
    display(pd.read_csv(sample_dist_files[0]).iloc[:8, :8])

if sample_dur_files:
    display(Markdown(f"### Preview sample duration matrix: `{sample_dur_files[0].name}`"))
    display(pd.read_csv(sample_dur_files[0]).iloc[:8, :8])

if sample_tt_files:
    display(Markdown(f"### Preview sample travel-time p85 matrix: `{sample_tt_files[0].name}`"))
    display(pd.read_csv(sample_tt_files[0]).iloc[:8, :8])

In [ ]:
# ============================================================
# CELL 11 — MANIFEST DAN ZIP AKHIR
# ============================================================

manifest = []
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "file": p.name,
            "relative_path": str(p.relative_to(OUTPUT_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 2)
        })

manifest_df = pd.DataFrame(manifest)
manifest_path = OUTPUT_DIR / "matrix_manifest.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

source_audit = pd.DataFrame({
    "Komponen": [
        "Excel source",
        "OSRM endpoint",
        "OSRM profile",
        "TomTom multiplier p50",
        "TomTom multiplier p85",
        "TomTom multiplier p95",
        "Selected dates processed",
        "Output root"
    ],
    "Nilai": [
        EXCEL_NAME,
        OSRM_BASE_URL,
        OSRM_PROFILE,
        TOMTOM_MULTIPLIER["p50"],
        TOMTOM_MULTIPLIER["p85"],
        TOMTOM_MULTIPLIER["p95"],
        len(selected_dates),
        str(OUTPUT_DIR)
    ]
})
source_audit_path = OUTPUT_DIR / "matrix_source_audit.csv"
source_audit.to_csv(source_audit_path, index=False, encoding="utf-8-sig")

display(Markdown("### Matrix Manifest"))
display(manifest_df.head(50))

display(Markdown("### Matrix Source Audit"))
display(source_audit)

zip_path = BASE_DIR / ZIP_NAME
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

print("ZIP berhasil dibuat:", zip_path)

if IN_COLAB:
    files.download(str(zip_path))